In [1]:
# Parameters
summary_config = {"run_run_comparison": False, "run_RTP_summary": False, "run_validation": True, "run_network_validation": False, "summary_list": {"RTP-summary-notebook": ["RTP_index", "RTP_congestion", "RTP_topsheet", "RTP_MIC", "RTP_person", "RTP_household", "RTP_access", "RTP_costs", "RTP_walk_bike", "RTP_emissions", "RTP_mode_share", "RTP_freight", "RTP_transit", "RTP_conformity_analysis"], "activitysim-validation-notebook": ["mandatory_tour_frequency", "non_mandatory_tour_frequency"], "network-validation-notebook": ["JBLM", "supplementals", "transit_validation", "traffic_validation", "bike_validation", "link_analysis"], "run-comparison-notebook": ["topsheet", "population", "parking", "vmt", "transit"]}, "p_output_dir": "outputs/summary", "output_folder": "outputs", "survey_folder": "inputs/base_year/survey", "uncloned_folder": "uncloned", "sc_run_name": "current run", "sc_run_path": "//modelstation3/c$/Workspace/sc_20251118/outputs", "survey_directories": {"survey": "R:/e2projects_two/2023_base_year/2023_survey/activitysim_format/skims_attached"}, "comparison_runs_list": {"2050 new transit, old network": "\\\\modelstation3\\c$\\Workspace\\sc_new_2050_transit\\soundcast", "2050 urbansim": "\\\\modelstation2\\c$\\Workspace\\sc_2050_urbansim2_07_30_25"}, "county_map": {"33": "King", "35": "Kitsap", "53": "Pierce", "61": "Snohomish"}, "uc_list": ["@sov_inc1", "@sov_inc2", "@sov_inc3", "@hov2_inc1", "@hov2_inc2", "@hov2_inc3", "@hov3_inc1", "@hov3_inc2", "@hov3_inc3", "@av_sov_inc1", "@av_sov_inc2", "@av_sov_inc3", "@av_hov2_inc1", "@av_hov2_inc2", "@av_hov2_inc3", "@av_hov3_inc1", "@av_hov3_inc2", "@av_hov3_inc3", "@tnc_inc1", "@tnc_inc2", "@tnc_inc3", "@mveh", "@hveh", "@bveh"], "agency_lookup": {"1": "King County Metro", "2": "Pierce Transit", "3": "Community Transit", "4": "Kitsap Transit", "5": "Washington Ferries", "6": "Sound Transit", "7": "Everett Transit"}, "emissions_scenario": "standard", "tot_veh_model_base_year": 3185281, "speed_bins": [-999999.0, 2.5, 7.5, 12.5, 17.5, 22.5, 27.5, 32.5, 37.5, 42.5, 47.5, 52.5, 57.5, 62.5, 67.5, 72.5, 999999.0], "fac_type_lookup": {"0": 0, "1": 4, "2": 4, "3": 5, "4": 5, "5": 5, "6": 3, "7": 5, "8": 0}, "tod_lookup": {"5to9": 7, "9to15": 12, "15to18": 17, "18to20": 19, "20to5": 22}, "summer_list": [87], "pollutant_map": {"1": "Total Gaseous HCs", "2": "CO", "3": "NOx", "5": "Methane", "6": "N20", "79": "Non-methane HCs", "87": "VOCs", "90": "Atmospheric CO2", "91": "Total Energy", "98": "CO2 Equivalent", "PM10": "PM10 Total", "PM25": "PM25 Total", "100": "PM10 Exhaust", "106": "PM10 Brakewear", "107": "PM10 Tirewear", "110": "PM25 Exhaust", "112": "Elemental Carbon", "115": "Sulfate Particulate", "116": "PM25 Brakewear", "117": "PM25 Tirewear", "118": "Composite NonECPM", "119": "H20 Aerosol"}, "special_route_lookup": {"1671": "A-Line Rapid Ride", "1672": "B-Line Rapid Ride", "1673": "C-Line Rapid Ride", "1674": "D-Line Rapid Ride", "1675": "E-Line Rapid Ride", "1677": "H-Line Rapid Ride", "4950": "Central Link", "6995": "Tacoma Link", "6998": "Sounder South", "6999": "Sounder North", "3701": "Swift Blue Line", "3702": "Swift Green Line"}}
input_config = {"debug_skims_and_paths": False, "model_year": "2023", "base_year": "2023", "landuse_inputs": "23_on_23_v3", "network_inputs": "network_2023_asim", "db_name": "soundcast_inputs_2023.db", "soundcast_inputs_dir": "R:/e2projects_two/SoundCast/Inputs/activitysim", "abm_model": "activitysim", "run_accessibility_calcs": False, "run_setup_emme_project_folders": False, "run_setup_emme_bank_folders": False, "run_copy_scenario_inputs": False, "run_import_networks": False, "run_skims_and_paths_free_flow": False, "run_skims_and_paths": False, "run_truck_model": False, "run_supplemental_trips": False, "run_abm": False, "run_summaries": True, "include_av": False, "include_tnc": True, "tnc_av": False, "include_tnc_to_transit": False, "include_knr_to_transit": False, "include_delivery": False, "include_telecommute": True, "run_integrated": False, "should_build_shadow_price": False, "delete_banks": False, "include_tnc_emissions": False, "add_distance_pricing": False, "distance_rate_dict": {"am": 13.5, "md": 8.5, "pm": 13.5, "ev": 8.5, "ni": 8.5}}


In [2]:
# import toml
# from pathlib import Path
# config_path = Path("../../../../configuration/asim_configuration")
# input_config = toml.load(config_path/ "input_configuration.toml")
# summary_config = toml.load(config_path/ "summary_configuration.toml")

In [3]:
import pandas as pd
import numpy as np
import plotly.express as px
import os, sys

module_path = os.path.abspath(os.path.join('../../../..')) # or the path to your source code
sys.path.insert(0, module_path)
import util

In [4]:
# read data
hh = util.get_hh_data(summary_config, uncloned=True)
person = util.get_person_data(summary_config, uncloned=True)
tour = util.get_tour_data(summary_config)

In [5]:
# count number of tours by tour category and type for each person
person_tour_cat = tour.groupby(['person_id','tour_cat_reassign','source'])['tour_id'].size().\
    unstack('tour_cat_reassign').reset_index()
person_tour_type = tour[~tour['tour_type'].isin(["work","school"])].groupby(['person_id','tour_type','source'], observed=True)['tour_id'].size().\
    unstack('tour_type').reset_index()
# cap tours at 4
for tour_cat in ['mandatory','non_mandatory']:
    person_tour_cat[tour_cat] = person_tour_cat[tour_cat].fillna(0)
    person_tour_cat[tour_cat] = person_tour_cat[tour_cat].apply(lambda x: "4+" if x>=4.0 else str(int(x)))

In [6]:
per_data = person.\
    merge(person_tour_cat, how='left', on=['person_id','source']).\
    merge(person_tour_type, how='left', on=['person_id','source']).\
    merge(hh[['household_id','income_group','num_drivers','auto_ownership','auto_ownership_4+','hhsize_4+','hh_weight','source']],
            how='left', on=['household_id','source']) # get auto ownership from hh data

non_mandatory_types = ['othmaint', 'shopping', 'escort', 'social', 'eatout', 'othdiscr', 'eat', 'business', 'maint']
per_data[non_mandatory_types] = per_data[non_mandatory_types].fillna(0)

tour_data = tour.merge(per_data, how='left', on=['person_id','household_id','source'])

## non-mandatory tours rates

In [7]:
# get tour rates
def calc_tour_rates(df_tour):
    """ Calculates tour rates.
    Parameters:
        df_tour: dataframe with tour data

    Returns:
        pandas dataframe for plotting
    """

    df_plot = df_tour.groupby(['source'],observed=True)[['tour_weight']].sum().reset_index(). \
        merge(per_data.groupby('source')['person_weight'].sum().reset_index(), how='left', on='source')
    df_plot['tour rate'] = df_plot['tour_weight']/df_plot['person_weight']

    return(df_plot)

In [8]:
df_plot = calc_tour_rates(tour_data[tour_data['tour_cat_reassign']=="non_mandatory"])
df_plot['all people'] = "non-mandatory"

fig = px.bar(df_plot, x="all people", y="tour rate", color="source",barmode="group",
             title="non-mandatory tour rates")
fig.update_layout(height=400, width=300)
fig.show()

In [9]:
# calculate tour rates by tour type
df_plot = pd.DataFrame()
for types in non_mandatory_types:
    df = calc_tour_rates(tour_data[tour_data['tour_type']==types])
    df['tour_type'] = types
    df_plot = pd.concat([df_plot,df])
df_plot['total number of tours'] = df_plot['tour_weight']

fig = px.bar(df_plot, x="tour_type", y="tour rate", color="source",barmode="group",
             hover_data=["total number of tours"],
             title="tour rates by tour type")
fig.update_layout(height=400, width=700)
fig.show()

## number of non-mandatory tours by segment

In [10]:
util.plot_share_barchart(per_data, 'person_weight', 'non_mandatory', 
                        "number of non mandatory tours in a day", height=400, width=700)

In [11]:
def plot_num_nonmand_tourtype(data, title):
    df_plot = pd.DataFrame()

    for type in non_mandatory_types:
        # calculate sample size and percentages
        df = data.groupby(['source', type], observed=True).\
            agg(sample_size=('person_weight', 'size'),
                weighted_sum=('person_weight', 'sum')).reset_index().\
            rename(columns={type: 'tour count'})
        
        df['percentage'] = df.groupby('source', group_keys=False, observed=True)['weighted_sum']. \
            apply(lambda x: x / float(x.sum()))
        df['tour type'] = type

        df_plot = pd.concat([df_plot,df])
     
    fig = px.bar(df_plot, x='tour count', y='percentage', color="source",barmode="group",
                 hover_data=["sample_size"],
                 facet_col='tour type', facet_col_wrap=3, orientation='v',
                 title=title)
    fig.for_each_annotation(lambda a: a.update(text = a.text.split("=")[-1]))
    fig.update_layout(height=500, width=750)
    fig.update_layout(yaxis1=dict(tickformat=".0%"), yaxis2=dict(tickformat=".0%"))

    fig.show()

plot_num_nonmand_tourtype(per_data, "number of non mandatory tours in a day by tour type")

### person type

In [12]:
util.plot_share_facetbar(per_data, 'person_weight', 'non_mandatory', "number of non-mandatory tours in a day by person type", 
                        'ptype_label', facet_col_wrap=3,
                        height=700, orientation='v')

In [13]:
plot_num_nonmand_tourtype(per_data[per_data['ptype']==1], "Full-time worker: number of non mandatory tours in a day by tour type")

In [14]:
plot_num_nonmand_tourtype(per_data[per_data['ptype']==2], "Part-time worker: number of non mandatory tours in a day by tour type")

In [15]:
plot_num_nonmand_tourtype(per_data[per_data['ptype']==3], "University student: number of non mandatory tours in a day by tour type")

In [16]:
plot_num_nonmand_tourtype(per_data[per_data['ptype']==4], "Non-working adult age < 65: number of non mandatory tours in a day by tour type")

In [17]:
plot_num_nonmand_tourtype(per_data[per_data['ptype']==5], "Non-working adult age 65+: number of non mandatory tours in a day by tour type")

### auto ownership

In [18]:
util.plot_share_facetbar(per_data, 'person_weight', 'non_mandatory', "number of non-mandatory tours in a day by auto ownership", 
                        'auto_ownership_4+', facet_col_wrap=5,
                        height=350, orientation='v')

In [19]:
plot_num_nonmand_tourtype(per_data[per_data['auto_ownership_4+']=="0"], "No cars in household: number of non mandatory tours in a day by tour type")

In [20]:
plot_num_nonmand_tourtype(per_data[per_data['auto_ownership'] < per_data['num_drivers']], "Fewer cars than drivers in household: number of non mandatory tours in a day by tour type")

### income

In [21]:
util.plot_share_facetbar(per_data, 'person_weight', 'non_mandatory', "number of non-mandatory tours in a day by income", 
                        'income_group', facet_col_wrap=5,
                        height=350, orientation='v')

In [22]:
plot_num_nonmand_tourtype(per_data[per_data['income_group'] == "low"], "Low income: number of non mandatory tours in a day by tour type")

In [23]:
plot_num_nonmand_tourtype(per_data[per_data['income_group'] == "medium"], "Medium income: number of non mandatory tours in a day by tour type")

In [24]:
plot_num_nonmand_tourtype(per_data[per_data['income_group'] == "medium-high"], "Medium-high income: number of non mandatory tours in a day by tour type")

In [25]:
plot_num_nonmand_tourtype(per_data[per_data['income_group'] == "high"], "High income: number of non mandatory tours in a day by tour type")